# User Behavior Prediction in Food Delivery Applications

This notebook follows the full machine learning workflow required by the project brief:

- load and inspect the data
- perform a first data audit and EDA
- explain feature engineering
- compare multiple models using ROC-AUC
- document the final ensemble pipeline


## 1. Imports and Setup

In [ ]:
from pathlib import Path

import pandas as pd

from src.baseline import (
    evaluate_catboost_baseline,
    evaluate_lightgbm_baseline,
    evaluate_logistic_baseline,
)
from src.data_utils import load_test_data, load_train_data
from src.features import build_features_v1


## 2. Load the Data

In [ ]:
train_df = load_train_data()
test_df = load_test_data()

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Train columns:', list(train_df.columns))
print('Test columns:', list(test_df.columns))

## 3. First Data Audit

This section checks the target balance and the main missing-value behavior.

In [ ]:
print('Target mean:', train_df['order_placed'].mean())

missing_summary = (
    train_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename('missing_fraction')
    .reset_index()
    .rename(columns={'index': 'column'})
)
missing_summary.head(10)

## 4. Feature Engineering

The final reproducible pipeline uses the frozen original feature set that produced the strongest public leaderboard behavior. These features are derived from existing columns rather than external data.

In [ ]:
feature_df = build_features_v1(train_df)
feature_df.head()

Examples of engineered features in the frozen pipeline:

- `session_duration_seconds`
- `active_span_seconds`
- `idle_gap_seconds`
- `avg_item_value`
- `declined_offer_ratio`
- `cart_to_min_order_gap`
- `meets_min_order`
- `discount_to_cart_ratio`
- `discount_to_min_order_ratio`

## 5. Model Comparison

The project compared multiple models using ROC-AUC and stratified cross-validation.

In [ ]:
logistic_result = evaluate_logistic_baseline(train_df, feature_builder=build_features_v1)
catboost_result = evaluate_catboost_baseline(train_df, feature_builder=build_features_v1)
lightgbm_result = evaluate_lightgbm_baseline(train_df, feature_builder=build_features_v1)

results_df = pd.DataFrame(
    [
        {'model': 'Logistic Regression', 'mean_roc_auc': logistic_result.mean_auc},
        {'model': 'CatBoost', 'mean_roc_auc': catboost_result.mean_auc},
        {'model': 'LightGBM', 'mean_roc_auc': lightgbm_result.mean_auc},
    ]
)
results_df.sort_values('mean_roc_auc', ascending=False)

## 6. Final Model Choice

The strongest competition approach was not a single model. The final method was a staged ensemble that:

1. built a leakage-safe target-encoded LightGBM model on the frozen original numeric feature set
2. rank-blended that target-encoded model with the previous best rank-ensemble winner
3. then rank-blended the two top-performing target-encoding blend winners 50/50
4. normalized the final scores into the `[0, 1]` range before saving the submission

This final ensemble achieved the strongest public leaderboard score of `0.97261` among all tested approaches.

## 7. Submission Generation

The final submission can be reproduced from the command below:

```powershell
python -m src.final_pipeline
```

This generates a CSV with columns `id` and `order_placed`.

## 8. Conclusion

Main takeaways:

- tree-based tabular models performed much better than the linear baseline
- LightGBM and CatBoost were the strongest individual models
- ensembling improved Kaggle performance further
- rank averaging generalized better than raw probability averaging for the final best method
